# Notebook 31d — Wu 2003 fault classification and parameter estimation with the pooled eta_col-widened posterior

**Not executed. Depends on notebook 24c having produced `wu2003_posterior_sb3_pooled_etawide.pkl`.**

Mirrors notebook 31b exactly (same taxonomy, same posterior-mass classifier, same parameter-estimation
and LDA sections), but loads the posterior trained in notebook 24c on the pooled
`alpha, beta_r, eta_col`-uniform bank instead of notebook 24b's `alpha, beta_r`-only-uniform bank.
Saves to distinctly-named output files so the two runs (`31b` vs `31d`) can be compared side by side
without overwriting each other.

Set `PRED_THRESH_OVERRIDE` below if notebook 31c's threshold-sensitivity experiment recommended a
different prediction threshold than the current default (`PRED_UNIT_THRESH_SB3=0.85`); leave it as
`None` to use the current default for a clean before/after comparison against notebook 31b.

In [ ]:
import pathlib, pickle, sys, time, importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    FAULT_UNITS_SB3, SB3_PARAM_NAMES, PRED_UNIT_THRESH_SB3,
    classify_fault_sb3, fault_unit_sb3_truth, fault_unit_sb3_predict, list_closed_loop_sb3,
)
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries

RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
FIGURES = ROOT / 'figures'; FIGURES.mkdir(exist_ok=True)
SBI_LOGS = ROOT / 'sbi-logs'
OI = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

PRED_THRESH_OVERRIDE = None  # set from notebook 31c's finding if it recommends a different threshold

with open(SBI_LOGS / 'wu2003_posterior_sb3_pooled_etawide.pkl', 'rb') as f:
    payload = pickle.load(f)
posterior = payload['posterior']
y0_sb = nominal_warm_start('S-B')
N_REPS = 30
N_POST = 200
print('using pooled eta_col-widened posterior:', {k: payload.get(k) for k in ['selected_seed', 'structure', 'bank', 'method']})

## Section 1 — Fault-unit taxonomy and posterior-mass classifier

Same taxonomy as notebook 31b (`[healthy, reactor, column, multi]`). If `PRED_THRESH_OVERRIDE` is
set, `classify_with_threshold` below is used instead of `classify_fault_sb3` so the threshold change
from notebook 31c can be tested together with the new posterior.

In [ ]:
def classify_with_threshold(samples, thresh):
    labels = [fault_unit_sb3_predict(row, thresh=thresh) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS_SB3}
    predicted = max(probs, key=probs.__getitem__)
    return {'class': predicted, 'probs': probs,
            'means': {p: float(np.mean(samples[:, i])) for i, p in enumerate(SB3_PARAM_NAMES)}}


def classify(samples):
    if PRED_THRESH_OVERRIDE is not None:
        return classify_with_threshold(samples, PRED_THRESH_OVERRIDE)
    return classify_fault_sb3(samples)


def compute_metrics(preds, trues, classes=FAULT_UNITS_SB3):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1 = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'confusion_matrix': cm, 'per_class_f1': f1, 'macro_f1': float(np.mean(list(f1.values()))), 'accuracy': float(np.trace(cm) / cm.sum())}

print(f'PRED_THRESH_OVERRIDE={PRED_THRESH_OVERRIDE} (None => using classify_fault_sb3 default {PRED_UNIT_THRESH_SB3})')

## Section 2 — Snapshot classification: posterior-mass method (12 scenarios x 30 replicates)

Identical protocol to notebook 31b, retained so the two runs are directly comparable.

In [ ]:
rows = []
all_preds, all_trues = [], []
all_summaries, all_labels, all_groups = [], [], []
t0 = time.time()
for sc in list_closed_loop_sb3():
    t_h, raw_obs = deterministic_window(sc, structure='S-B', y0=y0_sb)
    t_h_arr = np.asarray(t_h)
    rng = np.random.default_rng(310001 + sc.id)
    reps = noisy_replicates(np.asarray(raw_obs), n_replicates=N_REPS, rng=rng, noise_pct=0.003)
    true_theta = np.asarray(sc.theta())[:4]
    true_unit = fault_unit_sb3_truth(true_theta)
    rep_preds = []
    pooled_samples = []
    for i in range(N_REPS):
        s = compute_summaries(reps[i], 'S-B', t_h_arr)
        with torch.no_grad():
            samp = posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32), show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pooled_samples.append(samp)
        pred = classify(samp)
        rep_preds.append(pred['class'])
        all_preds.append(pred['class'])
        all_trues.append(true_unit)
        all_summaries.append(s)
        all_labels.append(true_unit)
        all_groups.append(sc.name)
    pooled_arr = np.concatenate(pooled_samples, axis=0)
    pooled = classify(pooled_arr)
    row = {'scenario': sc.name, 'true_unit': true_unit, 'pooled_pred': pooled['class'],
           'scenario_accuracy': float(np.mean([p == true_unit for p in rep_preds]))}
    for idx, name in enumerate(SB3_PARAM_NAMES):
        row[f'{name}_true'] = float(true_theta[idx])
        row[f'{name}_post_mean'] = float(pooled_arr[:, idx].mean())
    rows.append(row)
    print(f'{sc.name:24s} true={true_unit:8s} pooled={pooled["class"]:8s} scenario_acc={row["scenario_accuracy"]:.2f}')

metrics = compute_metrics(all_preds, all_trues)
results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS / '31d_fault_classification_metrics.csv', index=False)
summary_features_sb3 = np.asarray(all_summaries, dtype=np.float32)
print(f'\ndone in {time.time() - t0:.0f}s')
print('accuracy', metrics['accuracy'], 'macro_f1', metrics['macro_f1'])
print(metrics['per_class_f1'])

## Section 3 — Per-scenario table and parameter estimation accuracy

Combines notebook 31b's Table-9-equivalent and parameter-estimation sections. The key number to
watch against notebook 31b's baseline: `W12_snowball_compound`'s `eta_col_post_mean` should now sit
closer to its true 0.80 (was 0.867 in notebook 31b) if the pooled posterior actually fixed the
shrinkage that caused that scenario's misclassification.

In [ ]:
display_cols = (
    ['scenario', 'true_unit', 'pooled_pred', 'scenario_accuracy']
    + [f'{name}_true' for name in SB3_PARAM_NAMES]
    + [f'{name}_post_mean' for name in SB3_PARAM_NAMES]
)
print(results_df[display_cols].to_string(index=False))

est_rows = []
for name in SB3_PARAM_NAMES:
    true_vals = results_df[f'{name}_true'].values
    est_vals = results_df[f'{name}_post_mean'].values
    err = est_vals - true_vals
    est_rows.append({'parameter': name, 'bias': float(np.mean(err)), 'mae': float(np.mean(np.abs(err))), 'rmse': float(np.sqrt(np.mean(err ** 2)))})
est_df = pd.DataFrame(est_rows)
print()
print('=== Parameter estimation accuracy (posterior mean vs. true, 12 retained scenarios) ===')
print(est_df.to_string(index=False))
print()
w12 = results_df[results_df['scenario'] == 'W12_snowball_compound'].iloc[0]
print(f"W12_snowball_compound eta_col: true={w12['eta_col_true']:.3f}, post_mean={w12['eta_col_post_mean']:.3f} "
      f"(notebook 31b's pre-pooling estimate was 0.867)")

fig, axes = plt.subplots(2, 2, figsize=(9, 8), constrained_layout=True)
for ax, name in zip(axes.ravel(), SB3_PARAM_NAMES):
    true_vals = results_df[f'{name}_true'].values
    est_vals = results_df[f'{name}_post_mean'].values
    lo = min(true_vals.min(), est_vals.min()) - 0.03
    hi = max(true_vals.max(), est_vals.max()) + 0.03
    ax.plot([lo, hi], [lo, hi], color=OI[0], ls='--', lw=1.0)
    ax.scatter(true_vals, est_vals, color=OI[5], s=30, alpha=0.85, edgecolor='k', linewidth=0.4)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(f'true {name}'); ax.set_ylabel(f'posterior mean {name}'); ax.set_title(name)
    ax.grid(True, alpha=0.25)
plt.savefig(FIGURES / 'nb31d_parameter_estimation_parity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved nb31d_parameter_estimation_parity.png')

## Section 4 — Confusion matrix and macro-F1

In [ ]:
def plot_confusion_sb3(cm, title, path):
    fig, ax = plt.subplots(figsize=(5.8, 5.0), constrained_layout=True)
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(FAULT_UNITS_SB3))); ax.set_xticklabels(FAULT_UNITS_SB3, rotation=30, ha='right')
    ax.set_yticks(range(len(FAULT_UNITS_SB3))); ax.set_yticklabels(FAULT_UNITS_SB3)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            ax.text(j, i, str(val), ha='center', va='center', color='white' if val > cm.max() / 2 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()

plot_confusion_sb3(metrics['confusion_matrix'],
                   f'S-B3 pooled eta_col-widened posterior\nacc={metrics["accuracy"]:.2f}, macro-F1={metrics["macro_f1"]:.2f}',
                   FIGURES / '31d_sb3_confusion_matrix.png')

## Section 5 — LDA baseline (same protocol as notebook 31b, for reference)

In [ ]:
def run_lda_cv(X, y, groups):
    logo = LeaveOneGroupOut()
    y_pred, y_true = [], []
    for tr_i, te_i in logo.split(X, y, groups):
        if not np.all(np.isin(y[te_i], np.unique(y[tr_i]))):
            continue
        scaler = StandardScaler()
        lda = LinearDiscriminantAnalysis(solver='svd')
        lda.fit(scaler.fit_transform(X[tr_i]), y[tr_i])
        y_pred.extend(lda.predict(scaler.transform(X[te_i])))
        y_true.extend(y[te_i])
    return np.asarray(y_pred), np.asarray(y_true)

X_lda = summary_features_sb3
y_lda = np.asarray(all_labels)
groups_lda = np.asarray(all_groups)
lda_preds, lda_trues = run_lda_cv(X_lda, y_lda, groups_lda)
lda_metrics = compute_metrics(list(lda_preds), list(lda_trues))
print(f'LDA (Leave-One-Scenario-Out): accuracy={lda_metrics["accuracy"]:.3f}, macro_f1={lda_metrics["macro_f1"]:.3f}')
print(lda_metrics['per_class_f1'])

## Section 6 — Assessment: notebook 31b baseline vs. this pooled posterior

In [ ]:
print('=== Assessment: notebook 31b (baseline) vs. notebook 31d (pooled eta_col-widened) ===')
print(f'31b baseline : accuracy=0.831, macro-F1=0.793, per_class_f1={{"healthy": 0.667, "reactor": 0.855, "column": 1.000, "multi": 0.652}}')
print(f'31d (this run): accuracy={metrics["accuracy"]:.3f}, macro-F1={metrics["macro_f1"]:.3f}, per_class_f1={metrics["per_class_f1"]}')
print()
print('Key scenario to check: W12_snowball_compound was misclassified as reactor (0/30) in notebook 31b.')
w12_acc = results_df[results_df['scenario'] == 'W12_snowball_compound']['scenario_accuracy'].iloc[0]
print(f'W12_snowball_compound scenario_accuracy in this run: {w12_acc:.2f} (was 0.00 in notebook 31b)')

## Final acceptance criteria

Same bar as notebook 31b, applied to the pooled eta_col-widened posterior:
1. accuracy >= 0.85,
2. macro-F1 >= 0.85,
3. every retained fault unit has F1 >= 0.70,
4. no retained scenario has pooled/replicate accuracy below 0.50.

In [ ]:
criteria = [
    ('accuracy >= 0.85', float(metrics['accuracy']) >= 0.85, float(metrics['accuracy'])),
    ('macro-F1 >= 0.85', float(metrics['macro_f1']) >= 0.85, float(metrics['macro_f1'])),
    ('all retained class F1 >= 0.70', bool(all(v >= 0.70 for v in metrics['per_class_f1'].values())), metrics['per_class_f1']),
    ('no pooled scenario accuracy below 0.50', bool((results_df['scenario_accuracy'] >= 0.50).all()), results_df[['scenario', 'scenario_accuracy']].to_dict('records')),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")